# EdU Spheroid Analysis

CellProfiler output from a 3D spheroid assay measuring:
- **EdU incorporation** (DNA synthesis / S-phase) — diffuse nuclear signal
- **yH2AX foci** (DNA double-strand breaks) — discrete nuclear spots
- **HOECHST** (nuclear stain) — cell count reference

| Drug | Rows | Columns |
|---|---|---|
| Etoposide | A–D | 11–14 (bio replicates) |
| Olaparib  | E–H | 11–14 |
| 5-FU      | I–L | 11–14 |
| DMSO      | M–P | 11–14 |

Each well = one spheroid imaged at 10 z-planes (z0–z9, 5 µm spacing).

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (analysis_input, profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu

sns.set_style('white')
sns.set_context('notebook')
%matplotlib inline

DATA_DIR = external('spher_colo52_v1/3_Figure6/EdU')   # bulk per-object CSVs -- READ ONLY:
                                                       # another repository; nothing here writes to it
CACHED   = analysis_input('3_Figure6/EdU/data')   # archive -> derived -> committed
FIG_DIR  = figdir('Fig6')
FIG_DIR.mkdir(exist_ok=True)

DRUG_ORDER = ['Etoposide', 'Olaparib', '5-FU', 'DMSO']
# Matches the published panel, and the drug colours Fig 6c uses:
# 5-FU teal, olaparib purple, etoposide orange, DMSO grey.
PALETTE    = {'DMSO': '#c8c8c8', '5-FU': '#1B9E77',
              'Olaparib': '#5E3C99', 'Etoposide': '#E66101'}
Z_SPACING_UM = 5.0


def threshold_otsu(data):
    """Compute Otsu threshold on 1D array — no skimage required."""
    hist, edges = np.histogram(data, bins=256)
    hist = hist.astype(float) / hist.sum()
    centers = (edges[:-1] + edges[1:]) / 2
    w1 = np.cumsum(hist)
    w2 = 1.0 - w1
    mu1 = np.cumsum(hist * centers) / np.where(w1 > 0, w1, 1)
    mu_total = (hist * centers).sum()
    mu2 = (mu_total - np.cumsum(hist * centers)) / np.where(w2 > 0, w2, 1)
    variance = w1 * w2 * (mu1 - mu2) ** 2
    return centers[np.argmax(variance)]

## 1. Load data

In [ ]:
df_img    = pd.read_csv(DATA_DIR / 'EdU_Image.csv',          low_memory=False)
df_nuc    = pd.read_csv(DATA_DIR / 'EdU_nuclei.csv',         low_memory=False)
df_spots  = pd.read_csv(DATA_DIR / 'EdU_spots.csv',          low_memory=False)
df_nspots = pd.read_csv(DATA_DIR / 'EdU_nuclear_spots.csv',  low_memory=False)

print(f'Images : {len(df_img):>6,}')
print(f'Nuclei : {len(df_nuc):>6,}')
print(f'Spots  : {len(df_spots):>6,}')
print(f'NuclearSpots: {len(df_nspots):>6,}')

## 2. Plate metadata

In [ ]:
row_to_drug = {
    r: drug
    for drug, rows in [
        ('Etoposide', 'ABCD'),
        ('Olaparib',  'EFGH'),
        ('5-FU',      'IJKL'),
        ('DMSO',      'MNOP'),
    ]
    for r in rows
}

# Wells excluded due to poor clearing (spheroid not fully in imaging volume)
EXCLUDE_WELLS = ['L11', 'I14']

# z-plane encoded in filename (Frame_EdU column is always 0)
df_img['z_plane'] = df_img['FileName_EdU'].str.extract(r'-z(\d+)-').astype(int)
df_img['z_um']    = df_img['z_plane'] * Z_SPACING_UM
df_img['Drug']    = df_img['Metadata_Well'].str[0].map(row_to_drug)

print('Wells per drug (before exclusion):')
print(df_img.groupby('Drug')['Metadata_Well'].nunique().reindex(DRUG_ORDER))
print(f'\nExcluding wells with poor clearing: {EXCLUDE_WELLS}')

# Warn about unmapped wells
unmapped = df_img[df_img['Drug'].isna()]['Metadata_Well'].unique()
if len(unmapped):
    print('WARNING — unmapped wells:', unmapped)

## 3. Merge nuclei with image metadata

In [ ]:
df_nuc = df_nuc.merge(
    df_img[['ImageNumber', 'Metadata_Well', 'Drug', 'z_plane', 'z_um']],
    on='ImageNumber',
    how='left'
).copy()

# Apply well exclusions
df_nuc = df_nuc[~df_nuc['Metadata_Well'].isin(EXCLUDE_WELLS)]

print(f'Nuclei after exclusion: {len(df_nuc):,}  ({df_nuc["Drug"].notna().sum():,} with drug metadata)')

## 4. EdU positivity

EdU is a diffuse nuclear signal — cells in S-phase show elevated mean intensity.  
**The `spots` objects are yH2AX foci (DNA damage), not EdU spots.**

EdU+ classification uses Otsu's method on the log-transformed mean EdU intensity.  
Check the histogram below: the threshold should fall in the valley between the EdU− and EdU+ populations.

In [ ]:
edu_vals   = df_nuc['Intensity_MeanIntensity_EdU'].dropna().values
log_vals   = np.log1p(edu_vals)
otsu_log   = threshold_otsu(log_vals)
otsu_thresh = np.expm1(otsu_log)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log_vals, bins=150, color='steelblue', alpha=0.8)
ax.axvline(otsu_log, color='crimson', lw=2, linestyle='--',
           label=f'Otsu threshold (raw ≈ {otsu_thresh:.5f})')
ax.set_xlabel('log₁₊(EdU mean intensity)')
ax.set_ylabel('Nucleus count')
ax.set_title('EdU intensity distribution — verify bimodal split at threshold')
ax.legend()
sns.despine()
plt.tight_layout()
# [not a paper panel] fig.savefig(FIG_DIR / 'EdU_threshold_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Otsu threshold : {otsu_thresh:.6f}')

# ---- Adjust threshold here if needed ----
# otsu_thresh = 0.001   # override manually if Otsu misses the valley
# -----------------------------------------

df_nuc['EdU_positive'] = df_nuc['Intensity_MeanIntensity_EdU'] > otsu_thresh

n_pos = df_nuc['EdU_positive'].sum()
print(f'EdU+ nuclei    : {n_pos:,} / {len(df_nuc):,} ({100 * n_pos / len(df_nuc):.1f}%)')

## 5. Per-spheroid aggregation

Each well = one spheroid. All 10 z-planes are summed/aggregated to give one value per spheroid.

In [ ]:
# Nuclei in yH2AX-positive locations (for 3D foci spatial analysis)
df_foci = df_nuc[df_nuc['Children_spots_Count'] > 0]

# Main per-spheroid metrics
agg = df_nuc.groupby(['Metadata_Well', 'Drug']).agg(
    n_nuclei                = ('ObjectNumber',                   'count'),
    edu_positive_count      = ('EdU_positive',                   'sum'),
    median_edu_intensity    = ('Intensity_MeanIntensity_EdU',     'median'),
    n_yH2AX_foci            = ('Children_spots_Count',           'sum'),
    median_yH2AX_intensity  = ('Intensity_MeanIntensity_yH2AX',  'median'),
    median_hoechst          = ('Intensity_MeanIntensity_HOECHST', 'median'),
).reset_index()

# Derived metrics
agg['edu_fraction']          = agg['edu_positive_count'] / agg['n_nuclei']
agg['yH2AX_foci_per_nucleus'] = agg['n_yH2AX_foci'] / agg['n_nuclei']

# Mean 3D location of yH2AX foci (using nuclei that carry foci)
foci_loc = df_foci.groupby(['Metadata_Well', 'Drug']).agg(
    mean_foci_x    = ('Location_Center_X', 'mean'),
    mean_foci_y    = ('Location_Center_Y', 'mean'),
    mean_foci_z_um = ('z_um',              'mean'),
).reset_index()

agg = agg.merge(foci_loc, on=['Metadata_Well', 'Drug'], how='left')

# Fix Drug column ordering for plots
agg['Drug'] = pd.Categorical(agg['Drug'], categories=DRUG_ORDER, ordered=True)
agg = agg.sort_values('Drug')

print(f'Spheroids in aggregation table: {len(agg)}')
agg.head()

## 6. Spheroid detection

In [ ]:
expected_wells = {
    'Etoposide': [f'{r}{c}' for r in 'ABCD' for c in ['11','12','13','14']],
    'Olaparib':  [f'{r}{c}' for r in 'EFGH' for c in ['11','12','13','14']],
    '5-FU':      [f'{r}{c}' for r in 'IJKL' for c in ['11','12','13','14']],
    'DMSO':      [f'{r}{c}' for r in 'MNOP' for c in ['11','12','13','14']],
}

# Wells that actually had nuclei detected (at least 1 nucleus in any z-plane)
detected_wells = set(
    df_img[df_img['Count_nuclei'] > 0]['Metadata_Well']
)

rows = []
for drug in DRUG_ORDER:
    wells = expected_wells[drug]
    n_exp  = len(wells)
    n_det  = sum(1 for w in wells if w in detected_wells)
    rows.append({'Drug': drug, 'Expected': n_exp, 'Detected': n_det,
                 'Missing': n_exp - n_det,
                 'Detection rate': f'{100 * n_det / n_exp:.0f}%'})

detection_df = pd.DataFrame(rows)
detection_df.to_csv(CACHED / 'spheroid_detection.csv', index=False)
print(detection_df.to_string(index=False))

## 7. Visualizations

Each plot: bar (mean ± SEM) with beeswarm overlay (individual wells).  
Statistics: Kruskal-Wallis across all groups; if significant (p < 0.05), pairwise Mann-Whitney U vs DMSO with Bonferroni correction.  
Significance symbols: \* p < 0.05, \*\* p < 0.01, \*\*\* p < 0.001, ns = not significant.

In [ ]:
def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def plot_metric(df_sph, metric, ylabel, title):
    fig, ax = plt.subplots(figsize=(7, 5))

    # Bar (mean ± SEM)
    sns.barplot(
        data=df_sph, x='Drug', y=metric, order=DRUG_ORDER,
        palette=PALETTE, errorbar='se', ax=ax,
        alpha=0.55, capsize=0.08, err_kws={'linewidth': 1.5}
    )
    # Beeswarm overlay
    sns.stripplot(
        data=df_sph, x='Drug', y=metric, order=DRUG_ORDER,
        palette=PALETTE, ax=ax, size=6, jitter=True,
        linewidth=0.5, edgecolor='white', zorder=4
    )

    # Stats
    groups = [df_sph[df_sph['Drug'] == d][metric].dropna().values for d in DRUG_ORDER]
    groups_valid = [g for g in groups if len(g) > 1]
    stat_note = ''

    if len(groups_valid) >= 2:
        _, p_kw = kruskal(*groups_valid)
        stat_note = f'  (KW p={p_kw:.3g})'

        # Pairwise vs DMSO — Bonferroni corrected
        dmso_vals = df_sph[df_sph['Drug'] == 'DMSO'][metric].dropna().values
        if len(dmso_vals) > 1:
            drugs_vs = [d for d in DRUG_ORDER if d != 'DMSO']
            p_raw = []
            for d in drugs_vs:
                other = df_sph[df_sph['Drug'] == d][metric].dropna().values
                if len(other) > 1:
                    _, p = mannwhitneyu(other, dmso_vals, alternative='two-sided')
                else:
                    p = 1.0
                p_raw.append(p)
            # Bonferroni correction
            p_corr = [min(p * len(p_raw), 1.0) for p in p_raw]

            # Annotate above bars
            ymax = df_sph[metric].max()
            yrange = ymax - df_sph[metric].min()
            y_ann = ymax + yrange * 0.06
            for d, p in zip(drugs_vs, p_corr):
                x_pos = DRUG_ORDER.index(d)
                ax.text(x_pos, y_ann, sig_label(p),
                        ha='center', va='bottom', fontsize=11, color='black')

    ax.set_title(title + stat_note, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    sns.despine()
    plt.tight_layout()

    # [not a paper panel] per-metric diagnostic. Fig 6b is the three-panel pub_panel
    # figure further down, which the published figure shows as one panel.

    plt.show()
    plt.close()

In [ ]:
metrics = [
    ('n_nuclei',               'Nuclei count',                 'Number of nuclei per spheroid'),
    ('edu_positive_count',     'EdU+ cell count',              'Total EdU+ cells per spheroid'),
    ('edu_fraction',           'EdU+ fraction',                'EdU+ fraction per spheroid'),
    ('median_edu_intensity',   'Median EdU intensity',         'Median EdU intensity per nucleus'),
    ('n_yH2AX_foci',           'yH2AX foci count',             'Total yH2AX foci per spheroid'),
    ('yH2AX_foci_per_nucleus', 'yH2AX foci / nucleus',         'yH2AX foci per nucleus (DNA damage)'),
    ('median_yH2AX_intensity', 'Median yH2AX intensity',       'Median yH2AX intensity per nucleus'),
    ('median_hoechst',         'Median HOECHST intensity',     'Median HOECHST intensity per nucleus'),
]

for metric, ylabel, title in metrics:
    plot_metric(agg, metric, ylabel, title)

## 8. yH2AX foci — 3D spatial distribution

Each point = one spheroid (well).  
Coordinates: X and Y in pixels (1024×1024 image), Z in µm (z-plane × 5 µm).  
Only wells with detected foci are shown.  
Three projections give a sense of where in the spheroid DNA damage concentrates.

In [ ]:
agg_foci = agg.dropna(subset=['mean_foci_x', 'mean_foci_y', 'mean_foci_z_um'])

projections = [
    ('mean_foci_x', 'mean_foci_y',    'X (px)', 'Y (px)',   'XY projection'),
    ('mean_foci_x', 'mean_foci_z_um', 'X (px)', 'Z (µm)',   'XZ projection'),
    ('mean_foci_y', 'mean_foci_z_um', 'Y (px)', 'Z (µm)',   'YZ projection'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (xcol, ycol, xlabel, ylabel, subtitle) in zip(axes, projections):
    for drug in DRUG_ORDER:
        sub = agg_foci[agg_foci['Drug'] == drug]
        ax.scatter(sub[xcol], sub[ycol],
                   color=PALETTE[drug], label=drug,
                   s=60, alpha=0.85, linewidths=0.5, edgecolors='white')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'yH2AX foci centroids — {subtitle}')
    sns.despine(ax=ax)

axes[0].legend(title='Drug', bbox_to_anchor=(0, 1.02), loc='lower left',
               ncol=4, fontsize=9)
plt.tight_layout()
# [not a paper panel] fig.savefig(FIG_DIR / 'yH2AX_foci_3D_projections.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 9. Z-distribution of yH2AX foci

Boxplot of the mean foci Z-position per spheroid, per drug.  
Reveals whether DNA damage is concentrated at the spheroid surface (low/high Z) or core.

In [ ]:
plot_metric(
    agg.dropna(subset=['mean_foci_z_um']),
    'mean_foci_z_um',
    'Mean Z of yH2AX foci (µm)',
    'Axial position of yH2AX foci centroids'
)

## 10. Triangulated foci position within spheroid

Each nucleus is positioned relative to the spheroid's own centre of mass (mean X, Y, Z of all nuclei in that well). The 3D Euclidean distance is then normalised by the mean radius of that spheroid, giving a **relative radial position** (0 = centre, 1 = edge) that is comparable across spheroids of different sizes.

Pixel size assumed: **0.65 µm** (20× objective). Adjust `PX_UM` below if needed.

In [ ]:
PX_UM = 0.65   # µm per pixel at 20× — adjust if needed

# Spheroid centroid per well (all nuclei)
centroid = df_nuc.groupby('Metadata_Well').agg(
    cx=('Location_Center_X', 'mean'),
    cy=('Location_Center_Y', 'mean'),
    cz=('z_um',              'mean'),
).reset_index()

df_nuc2 = df_nuc.merge(centroid, on='Metadata_Well', how='left')

# 3D distance from centroid for every nucleus (in µm)
df_nuc2['dx_um'] = (df_nuc2['Location_Center_X'] - df_nuc2['cx']) * PX_UM
df_nuc2['dy_um'] = (df_nuc2['Location_Center_Y'] - df_nuc2['cy']) * PX_UM
df_nuc2['dz_um'] =  df_nuc2['z_um'] - df_nuc2['cz']
df_nuc2['dist_um'] = np.sqrt(df_nuc2['dx_um']**2 + df_nuc2['dy_um']**2 + df_nuc2['dz_um']**2)

# Mean spheroid radius per well
radius = df_nuc2.groupby('Metadata_Well')['dist_um'].mean().rename('radius_um').reset_index()

# Foci-bearing nuclei only
df_foci2 = df_nuc2[df_nuc2['Children_spots_Count'] > 0]

foci_radial = df_foci2.groupby(['Metadata_Well', 'Drug']).agg(
    mean_foci_dist=('dist_um', 'mean'),
).reset_index().merge(radius, on='Metadata_Well')

foci_radial['rel_dist'] = foci_radial['mean_foci_dist'] / foci_radial['radius_um']
foci_radial['Drug'] = pd.Categorical(foci_radial['Drug'], categories=DRUG_ORDER, ordered=True)

print('Relative radial position of yH2AX foci per drug (mean ± std):')
print(foci_radial.groupby('Drug')['rel_dist'].agg(['median','mean','std']).round(3).reindex(DRUG_ORDER))

In [ ]:
# Beeswarm + bar of relative radial position
plot_metric(foci_radial, 'rel_dist',
            'Relative radial position (0=centre, 1=edge)',
            'yH2AX foci — triangulated position within spheroid')

# Also absolute distance
plot_metric(foci_radial, 'mean_foci_dist',
            'Mean distance from spheroid centroid (µm)',
            'yH2AX foci — absolute distance from spheroid centre')

## 10. Summary table

In [ ]:
summary_cols = [
    'n_nuclei', 'edu_positive_count', 'edu_fraction',
    'n_yH2AX_foci', 'yH2AX_foci_per_nucleus',
]

summary = (
    agg.groupby('Drug')[summary_cols]
    .agg(['mean', 'std'])
    .round(3)
    .reindex(DRUG_ORDER)
)

summary.to_csv(CACHED / 'spheroid_summary.csv')
summary

## 11. Publication panels A & B

**Panel A** — γH2AX foci / nucleus: ascending-damage order anchored by etoposide.  
**Panel B** — EdU⁺ fraction: same x-axis, showing the inverse proliferation gradient.  
X-axis order (left → right): DMSO → 5-FU → Olaparib → Etoposide.

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    'font.family':        'sans-serif',
    'font.sans-serif':    ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          6,
    'axes.labelsize':     6,
    'xtick.labelsize':    6,
    'ytick.labelsize':    6,
    'axes.linewidth':     0.5,
    'xtick.major.width':  0.5,
    'ytick.major.width':  0.5,
    'xtick.major.size':   2,
    'ytick.major.size':   2,
    'lines.linewidth':    0.8,
    'pdf.fonttype':       42,
    'svg.fonttype':       'none',
})

PUB_ORDER = ['DMSO', '5-FU', 'Olaparib', 'Etoposide']
# Drug colours as published, and the same scheme Fig 6c uses: DMSO grey, 5-FU teal,
# olaparib purple, etoposide orange. This was a four-step greyscale.
PUB_PAL   = {'DMSO': '#c8c8c8', '5-FU': '#1B9E77',
             'Olaparib': '#5E3C99', 'Etoposide': '#E66101'}

FIG_W = 3.5


def sig_bracket(ax, x1, x2, y, tick, label):
    ax.plot([x1, x1, x2, x2], [y - tick, y, y, y - tick],
            color='black', lw=0.5, clip_on=False)
    ax.text((x1 + x2) / 2, y + tick * 0.3, label,
            ha='center', va='bottom', fontsize=6)


def pub_panel(ax, df_sph, metric, ylabel, panel_label=None, extra_pair=None):
    sns.swarmplot(
        data=df_sph, x='Drug', y=metric, order=PUB_ORDER,
        hue='Drug', hue_order=PUB_ORDER, palette=PUB_PAL,
        legend=False, ax=ax, size=2,
        linewidth=0.25, edgecolor='white', zorder=5,
    )

    for xi, d in enumerate(PUB_ORDER):
        vals = df_sph[df_sph['Drug'] == d][metric].dropna().values
        if len(vals) == 0:
            continue
        mean = vals.mean()
        sem  = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
        ax.plot([xi - 0.20, xi + 0.20], [mean, mean],
                color='black', lw=1.0, solid_capstyle='round', zorder=6)
        ax.errorbar(xi, mean, yerr=sem,
                    color='black', lw=0.6, capsize=2, capthick=0.6, zorder=6)

    dmso   = df_sph[df_sph['Drug'] == 'DMSO'][metric].dropna().values
    others = [d for d in PUB_ORDER if d != 'DMSO']
    p_raw  = []
    for d in others:
        v = df_sph[df_sph['Drug'] == d][metric].dropna().values
        _, p = mannwhitneyu(v, dmso, alternative='two-sided') if (len(v) > 1 and len(dmso) > 1) else (0, 1.0)
        p_raw.append(p)
    p_corr = [min(p * len(others), 1.0) for p in p_raw]

    ymax   = df_sph[metric].max()
    ymin   = df_sph[metric].min()
    y_step = max((ymax - ymin) * 0.08, ymax * 0.05)

    for d, p in zip(others, p_corr):
        ax.text(PUB_ORDER.index(d), ymax + y_step,
                sig_label(p), ha='center', va='bottom', fontsize=6)

    if extra_pair is not None:
        d1, d2 = extra_pair
        v1 = df_sph[df_sph['Drug'] == d1][metric].dropna().values
        v2 = df_sph[df_sph['Drug'] == d2][metric].dropna().values
        if len(v1) > 1 and len(v2) > 1:
            _, p_pair = mannwhitneyu(v1, v2, alternative='two-sided')
            x1, x2   = PUB_ORDER.index(d1), PUB_ORDER.index(d2)
            sig_bracket(ax, x1, x2, ymax + y_step * 2.6, y_step * 0.35, sig_label(p_pair))

    ax.set_ylim(bottom=0, top=ymax + y_step * 5.0)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel, fontsize=6)
    ax.set_xticks(range(len(PUB_ORDER)))
    ax.set_xticklabels(PUB_ORDER, rotation=45, ha='right', fontsize=6,
                       rotation_mode='anchor')
    sns.despine(ax=ax, offset=2, trim=True)
    # after despine(trim=True), which rebuilds the ticks and drops the rotation
    # set on set_xticklabels above
    ax.tick_params(axis='x', rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha('right'); lbl.set_rotation_mode('anchor')

    if panel_label:
        # the paper titles these "DNA Damage" / "S-phase entry" / "Size" rather than
        # labelling them with a bold corner letter
        ax.set_title(panel_label, fontsize=8, pad=6)


# wider than FIG_W=3.5in: three panels of four rotated labels need the room
fig, axes = plt.subplots(1, 3, figsize=(FIG_W * 1.9, 2.6))

pub_panel(axes[0], agg, 'yH2AX_foci_per_nucleus', 'γH2AX foci / nucleus', 'DNA Damage',
          extra_pair=('5-FU', 'Olaparib'))
pub_panel(axes[1], agg, 'edu_fraction',            'EdU+ fraction',         'S-phase entry',
          extra_pair=('5-FU', 'Olaparib'))
pub_panel(axes[2], agg, 'n_nuclei',                'Nuclei / spheroid',     'Size',
          extra_pair=('5-FU', 'Olaparib'))

# Shared footnote with n per group
n_per  = {d: int(agg[agg['Drug'] == d]['n_nuclei'].dropna().shape[0]) for d in PUB_ORDER}
footnote = '  '.join([f'{d}, n={n_per[d]}' for d in PUB_ORDER])
plt.tight_layout(w_pad=1.0)
fig.text(0.5, -0.02, footnote, ha='center', va='top', fontsize=6, color='#444444')

# Fig 6b is this three-panel figure, saved below once panel_source exists.
plt.show()
plt.close()
print('Saved → figures/panels_AB.png  and  .pdf')

# ---- Source data behind publication panels A/B/C (one row per spheroid) ----
PANEL_METRICS = ['yH2AX_foci_per_nucleus', 'edu_fraction', 'n_nuclei']
panel_source = (
    agg.loc[agg['Drug'].isin(PUB_ORDER),
            ['Metadata_Well', 'Drug', *PANEL_METRICS]]
    .assign(Drug=lambda d: pd.Categorical(d['Drug'], categories=PUB_ORDER, ordered=True))
    .sort_values(['Drug', 'Metadata_Well'])
    .reset_index(drop=True)
)
panel_source.to_csv(CACHED / 'panel_source_data.csv', index=False)
print(f'Saved → panel_source_data.csv  ({len(panel_source)} spheroids, '
      f'columns: {", ".join(PANEL_METRICS)})')

# The published Fig 6b is these three panels together, with mean +/- SEM and the
# significance brackets -- not the per-metric bar/swarm diagnostics above.
save_panel(fig, 'Fig6b', data=panel_source,
           caption='DNA damage, S-phase entry and spheroid size by treatment',
           notebook='analysis/3_Figure6/EdU/EdU_analysis.ipynb')

In [ ]:
from scipy.stats import mannwhitneyu, ttest_ind

fu_edu    = agg[agg['Drug'] == '5-FU']['edu_fraction'].dropna().values
olap_edu  = agg[agg['Drug'] == 'Olaparib']['edu_fraction'].dropna().values

stat_mw, p_mw   = mannwhitneyu(fu_edu, olap_edu, alternative='less')   # one-sided: 5-FU < Olaparib
stat_t,  p_t    = ttest_ind(fu_edu, olap_edu, alternative='less')       # parametric for comparison

print('5-FU vs Olaparib — EdU+ fraction')
print(f'  5-FU     : n={len(fu_edu)},  mean={fu_edu.mean():.3f},  median={np.median(fu_edu):.3f},  SD={fu_edu.std(ddof=1):.3f}')
print(f'  Olaparib : n={len(olap_edu)}, mean={olap_edu.mean():.3f},  median={np.median(olap_edu):.3f},  SD={olap_edu.std(ddof=1):.3f}')
print()
print(f'  Mann-Whitney U (one-sided, H1: 5-FU < Olaparib): U={stat_mw:.0f}, p={p_mw:.4f}')
print(f'  Welch t-test   (one-sided, H1: 5-FU < Olaparib): t={stat_t:.3f}, p={p_t:.4f}')

In [ ]:
# 5-FU vs Olaparib — yH2AX foci/nucleus and nuclei/spheroid (size)
# Mirrors the EdU cell above; adds effect sizes (Cliff's delta for MW, Cohen's d for t).

def cliffs_delta(a, b):
    """Non-parametric effect size: (#a>b - #a<b)/(n_a*n_b), range [-1, 1]."""
    diff = a[:, None] - b[None, :]
    return (np.sum(diff > 0) - np.sum(diff < 0)) / (len(a) * len(b))

def cohens_d(a, b):
    """Standardised mean difference using pooled SD."""
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp

def compare(metric, label, a_drug='5-FU', b_drug='Olaparib'):
    a = agg[agg['Drug'] == a_drug][metric].dropna().values
    b = agg[agg['Drug'] == b_drug][metric].dropna().values

    # Two-sided (reviewer-safe); use alternative='less'/'greater' for directional H1.
    U, p_mw = mannwhitneyu(a, b, alternative='two-sided')
    t, p_t  = ttest_ind(a, b, alternative='two-sided')   # Welch

    print(f'{a_drug} vs {b_drug} — {label}')
    print(f'  {a_drug:9}: n={len(a)},  mean={a.mean():.3f},  median={np.median(a):.3f},  SD={a.std(ddof=1):.3f}')
    print(f'  {b_drug:9}: n={len(b)},  mean={b.mean():.3f},  median={np.median(b):.3f},  SD={b.std(ddof=1):.3f}')
    print(f'  Mann-Whitney U (two-sided): U={U:.0f}, p={p_mw:.4g},  Cliff delta={cliffs_delta(a, b):+.3f}')
    print(f'  Welch t-test   (two-sided): t={t:.3f}, p={p_t:.4g},  Cohen d={cohens_d(a, b):+.3f}')
    print()

compare('yH2AX_foci_per_nucleus', 'yH2AX foci / nucleus')
compare('n_nuclei',               'Nuclei / spheroid (size)')
